# Phase 3 — Fine-tune on IndicDLP (Colab)

Picks up from the Phase 2 pretrained 9-class checkpoint and runs:
1. **Pseudo-labeling** of BaDLAD-unlabeled with the pretrained model
2. **CBST** class-balanced thresholds (rare classes not swamped)
3. **Self-training rounds** on {real + pseudo}
4. **Fine-tune on IndicDLP** (re-headed to IndicDLP's ontology) — *added next*

Logic lives in `src/finetuning/self_training.py`; these cells only orchestrate.
**Runtime:** T4 is fine for the smoke test (Cell 4). Switch to **A100** for the full run (Cell 5), off-peak.

## Cell 0 — Bootstrap (mount Drive, install deps)

In [1]:
# Runtime -> Change runtime type -> A100 (full run) or T4 (smoke test) -> Save FIRST
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


subprocess.run(['pip','install','-q','ultralytics','huggingface_hub','pyyaml',
                'pycocotools','kagglehub','tqdm'], check=True)
subprocess.run(['pip','install','-q',
                'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT_ROOT:', PROJECT_ROOT)

Mounted at /content/drive
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA RTX PRO 6000 Blackwell Server Edition
PROJECT_ROOT: /content/drive/MyDrive/doclayout-yolo-indic


## Cell 1 — Sync code from GitHub (plain files, no zip)
Clones the repo, auto-detects `src/` (works whether it's at the repo root or inside a
wrapper dir like `doclayout-yolo-indic/`), and copies it to `PROJECT_ROOT/src` on Drive so
`config.py`'s `__file__`-anchored paths resolve to Drive (persistent outputs/checkpoints).
**Prereq:** commit `src/` as plain files (drop the zip) and add
`src/finetuning/self_training.py` + `__init__.py`.

In [2]:
import subprocess, shutil
from pathlib import Path

GITHUB_URL    = 'https://github.com/vigneshpalanivelr/mtech-project-aiml.git'
GITHUB_BRANCH = 'main'
REPO_LOCAL    = Path('/content/_repo')

shutil.rmtree(REPO_LOCAL, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','-b',GITHUB_BRANCH,
                GITHUB_URL, str(REPO_LOCAL)], check=True)

# Find src/ wherever it lives: repo root, or one level down (wrapper dir).
candidates = [REPO_LOCAL/'src'] + sorted(REPO_LOCAL.glob('*/src'))
SRC = next((c for c in candidates if (c/'config.py').exists()), None)
assert SRC, f'Could not find src/config.py under {REPO_LOCAL}'
BASE = SRC.parent
print('Found project at:', BASE)

# Copy code (not docs) to PROJECT_ROOT so REPO_ROOT = parents[1] -> Drive.
for item in ['src','tests','requirements.txt','README.md']:
    s = BASE/item; d = PROJECT_ROOT/item
    if s.is_dir():   shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
    elif s.exists(): shutil.copy(s, d)

assert (PROJECT_ROOT/'src'/'finetuning'/'self_training.py').exists(), \
    'Commit src/finetuning/self_training.py + __init__.py to the repo first!'
print('Code synced (plain files) ->', PROJECT_ROOT/'src')

Found project at: /content/_repo/doclayout-yolo-indic
Code synced (plain files) -> /content/drive/MyDrive/doclayout-yolo-indic/src


## Cell 6 - IndicDLP class check

In [3]:
%cd {PROJECT_ROOT}
from src.finetuning.train_finetuning import stage_indicdlp, read_classes

# need_train=0 -> download val fully, skip train entirely (just the class check)
raw = stage_indicdlp('VigneshPR/IndicDLP', '/content/IndicDLP_raw', need_train=0)
names, id_to_idx = read_classes(raw/'annotations'/'instances_val2017.json')
print(f"\n{len(names)} classes:")
print(names)

/content/drive/MyDrive/doclayout-yolo-indic


annotations/instances_train2017.json:   0%|          | 0.00/378M [00:00<?, ?B/s]

annotations/instances_val2017.json:   0%|          | 0.00/47.1M [00:00<?, ?B/s]

16:31:00 | INFO    | doclayout_indic.train_finetuning | Downloading val2017.tar ...


val2017.tar:   0%|          | 0.00/5.39G [00:00<?, ?B/s]

KeyboardInterrupt: 

## Cell 7 - Fine-tune on IndicDLP

In [ ]:
%cd {PROJECT_ROOT}
from src.finetuning.train_finetuning import run_finetuning

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

SELF_TRAINED = PROJECT_ROOT/'output'/'self_training_real'/'round_2'/'train'/'weights'/'best.pt'
assert SELF_TRAINED.exists(), f"self-trained checkpoint not found: {SELF_TRAINED}"

best = run_finetuning(
    self_trained = SELF_TRAINED,
    hf_repo      = 'VigneshPR/IndicDLP',
    drive_out    = PROJECT_ROOT/'data'/'indicdlp_yolo',
    local_root   = '/content/indicdlp_yolo',
    work_dir     = PROJECT_ROOT/'output'/'finetune_indicdlp',
    train_cap    = 20000,   # subset of 95K for compute
)
print('Final Phase 3 model:', best)

## Next
- `src/finetuning/data_prep.py` — BaDLAD COCO->YOLO 9-class remap + IndicDLP yaml
- `src/finetuning/train_finetuning.py` — re-head to IndicDLP's 42 classes + fine-tune (final model)
- `src/finetuning/ablation.py` — the 3-5 ablation runs

(`src/utils/ontology.py` already exists: pretrain 9-class head, then **replace the head**
with IndicDLP's 42 classes at fine-tuning. Run its `inspect_indicdlp_categories()` first.)

## Cell 8 - Backup the checkpoint

In [ ]:
from pathlib import Path
from huggingface_hub import upload_file
from google.colab import userdata
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')

best = PROJECT_ROOT/'output'/'finetune_indicdlp'/'finetune'/'weights'/'best.pt'

assert best.exists(), f"NOT FOUND: {best}"
print(f"Found: {best}  ({best.stat().st_size/1e6:.1f} MB)")

upload_file(
    path_or_fileobj=str(best),
    path_in_repo="phase3/indicdlp_finetuned_42class.pt",
    repo_id="VigneshPR/indicsynth-150k", repo_type="dataset",
    token=userdata.get('HF_TOKEN'),
)
print("Backed up to HF.")

In [4]:
import pandas as pd
from pathlib import Path

def summarize(tag, csv_path):
    p = Path(csv_path)
    if not p.exists():
        print(f"\n[{tag}] results.csv NOT found at {p}")
        return
    df = pd.read_csv(p); df.columns = df.columns.str.strip()
    cols = [c for c in ['epoch','metrics/mAP50(B)','metrics/mAP50-95(B)',
                        'metrics/precision(B)','metrics/recall(B)'] if c in df.columns]
    print(f"\n===== {tag} =====")
    print(df[cols].to_string(index=False))
    # best epoch by mAP50-95
    if 'metrics/mAP50-95(B)' in df.columns:
        b = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
        print(f"  BEST epoch {int(b['epoch'])}: "
              f"mAP50={b['metrics/mAP50(B)']:.3f}  mAP50-95={b['metrics/mAP50-95(B)']:.3f}")

base = PROJECT_ROOT/'output'/'self_training_real'
summarize("Self-training Round 1", base/'round_1'/'train'/'results.csv')
summarize("Self-training Round 2", base/'round_2'/'train'/'results.csv')

# also the fine-tuning run, for the full picture
summarize("Fine-tune on IndicDLP", PROJECT_ROOT/'output'/'finetune_indicdlp'/'finetune'/'results.csv')


===== Self-training Round 1 =====
 epoch  metrics/mAP50(B)  metrics/mAP50-95(B)  metrics/precision(B)  metrics/recall(B)
     1           0.55516              0.38126               0.59317            0.57236
     2           0.58664              0.41507               0.63583            0.58502
     3           0.63513              0.45724               0.64713            0.64759
     4           0.68826              0.51479               0.74232            0.62752
     5           0.69771              0.51619               0.76721            0.65742
     6           0.68931              0.51546               0.75112            0.65424
     7           0.69519              0.51429               0.77049            0.65505
  BEST epoch 5: mAP50=0.698  mAP50-95=0.516

===== Self-training Round 2 =====
 epoch  metrics/mAP50(B)  metrics/mAP50-95(B)  metrics/precision(B)  metrics/recall(B)
     1           0.66172              0.49328               0.71080            0.63789
     2          

## Cell 9 - Ablation

In [3]:
%cd {PROJECT_ROOT}
from src.finetuning.train_finetuning import run_finetuning
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

# ABLATION BASELINE: fine-tune the PRETRAINED-ONLY checkpoint [B]
# (skips self-training entirely). Everything else identical to the full run,
# so the mAP gap vs your 0.328 isolates self-training's contribution.
PRETRAINED_B = PROJECT_ROOT/'output'/'checkpoints'/'doclayout_yolo_indic_pretrained.pt'
assert PRETRAINED_B.exists(), f"checkpoint [B] not found: {PRETRAINED_B}"

best_ablation = run_finetuning(
    self_trained = PRETRAINED_B,                               # <-- [B], not round_2
    hf_repo      = 'VigneshPR/IndicDLP',
    drive_out    = PROJECT_ROOT/'data'/'indicdlp_yolo',         # same cached dataset (reused)
    local_root   = '/content/indicdlp_yolo',
    work_dir     = PROJECT_ROOT/'output'/'ablation_no_selftrain',   # <-- separate folder
    train_cap    = 20000,   # IDENTICAL to the full run
)
print('Ablation (no self-training) model:', best_ablation)

/content/drive/MyDrive/doclayout-yolo-indic
16:56:19 | INFO    | doclayout_indic.train_finetuning | Reusing Drive cache (train 12080 / val 3000 imgs) -> local
17:10:20 | INFO    | doclayout_indic.train_finetuning | Wrote /content/drive/MyDrive/doclayout-yolo-indic/data/indicdlp_yolo/indicdlp.yaml (nc=42)
17:10:20 | INFO    | doclayout_indic.train_finetuning | Fine-tuning doclayout_yolo_indic_pretrained.pt on IndicDLP (42 classes)
New https://pypi.org/project/doclayout_yolo/0.0.4 available 😃 Update with 'pip install -U doclayout_yolo'
Ultralytics YOLOv0.0.2 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/doclayout-yolo-indic/output/checkpoints/doclayout_yolo_indic_pretrained.pt, data=/content/drive/MyDrive/doclayout-yolo-indic/data/indicdlp_yolo/indicdlp.yaml, epochs=30, time=None, patience=8, batch=16, imgsz=1024, save=True, save_period=1, val_period=1, cache=False,

/usr/local/lib/python3.12/dist-packages/doclayout_yolo/utils/checks.py:641: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):


AMP: checks passed ✅


/usr/local/lib/python3.12/dist-packages/doclayout_yolo/engine/trainer.py:277: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning /content/indicdlp_yolo/labels/train2017... 12080 images, 56 backgrounds, 0 corrupt: 100%|██████████| 12080/12080 [00:01<00:00, 6456.04it/s]

train: WARNING ⚠️ /content/indicdlp_yolo/images/train2017/br_pa_000109_0.jpg: 1 duplicate labels removed


...

3000
                   all       3000      46584      0.544      0.504       0.51      0.355

      Epoch    GPU_mem     box_om     cls_om     dfl_om     box_oo     cls_oo     dfl_oo  Instances       Size


      30/30      42.2G     0.8011     0.7663      1.057     0.8883     0.9308      1.078        299       1024: 100%|██████████| 755/755 [04:06<00:00,  3.07it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:22<00:00,  4.09it/s]


3000
                   all       3000      46584      0.563      0.494      0.511      0.356

30 epochs completed in 2.323 hours.
Optimizer stripped from /content/drive/MyDrive/doclayout-yolo-indic/output/ablation_no_selftrain/finetune/weights/last.pt, 40.7MB
Optimizer stripped from /content/drive/MyDrive/doclayout-yolo-indic/output/ablation_no_selftrain/finetune/weights/best.pt, 40.7MB

Validating /content/drive/MyDrive/doclayout-yolo-indic/output/ablation_no_selftrain/finetune/weights/best.pt...
Ultralytics YOLOv0.0.2 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:26<00:00,  3.56it/s]


3000
                   all       3000      46584      0.563      0.494      0.511      0.356
         advertisement       3000        408      0.615      0.792      0.734      0.635
                answer       3000        129      0.585      0.142      0.169      0.132
                author       3000        602      0.582      0.424      0.466      0.315
         chapter-title       3000        186      0.485      0.435      0.475      0.304
          contact-info       3000        430      0.493      0.312      0.325      0.203
              dateline       3000        977      0.668      0.532      0.589      0.325
                figure       3000       2376      0.736      0.793      0.808      0.667
        figure-caption       3000        678      0.574      0.544      0.547      0.346
  first-level-question       3000       1172      0.585      0.629      0.615      0.495
                  flag       3000        116      0.517      0.509       0.54      0.338
                

In [5]:
from huggingface_hub import upload_file
from google.colab import userdata
upload_file(
    path_or_fileobj=str(PROJECT_ROOT/'output'/'ablation_no_selftrain'/'finetune'/'weights'/'best.pt'),
    path_in_repo="phase3/ablation_no_selftrain_42class.pt",
    repo_id="VigneshPR/indicsynth-150k", repo_type="dataset",
    token=userdata.get('HF_TOKEN'),
)
print("Ablation checkpoint backed up to HF.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../finetune/weights/best.pt:   0%|          |  160kB / 40.7MB            

Ablation checkpoint backed up to HF.


In [6]:
from pathlib import Path
from google.colab import runtime

ab = PROJECT_ROOT/'output'/'ablation_no_selftrain'/'finetune'
checks = {
    'ablation best.pt':   (ab/'weights'/'best.pt').exists(),
    'ablation results':   (ab/'results.csv').exists(),
    'final model [D]':    (PROJECT_ROOT/'output'/'finetune_indicdlp'/'finetune'/'weights'/'best.pt').exists(),
    'pretrained [B]':     (PROJECT_ROOT/'output'/'checkpoints'/'doclayout_yolo_indic_pretrained.pt').exists(),
}
for k, ok in checks.items():
    print(('OK   ' if ok else 'MISSING  ') + k)

# confirm the ablation actually reached the end, not mid-run
import pandas as pd
n = len(pd.read_csv(ab/'results.csv')) if (ab/'results.csv').exists() else 0
print(f"\nAblation epochs completed: {n} / 30")

if all(checks.values()) and n >= 30:
    print("\nAll safe on Drive. Releasing runtime in 5s...")
    import time; time.sleep(5)
    runtime.unassign()
else:
    print("\nNOT releasing — ablation not finished (or something missing above).")

OK   ablation best.pt
OK   ablation results
OK   final model [D]
OK   pretrained [B]

Ablation epochs completed: 30 / 30

All safe on Drive. Releasing runtime in 5s...


RuntimeManagementError: Unable to request VM unassignment.